# Limpeza de Dados

Este notebook parte do arquivo bruto e produz o dataset limpo `dados_limpos.xlsx`.

Pré-requisito: executar `reconhecimento_dados.ipynb` para entender a estrutura do arquivo.

Etapas:
1. Limpeza Dataset: remover colunas nao utilizaveis na analise, converter datas, criar flags, padronizar textos, renomear colunas, padronizar categorias, remover duplicatas.
2. Junção das três abas
3. Resumo final de qualidade do novo dataset limpo
4. Exportação

In [38]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FILE = 'Dados - Atividade 1.xlsx'

df_main = pd.read_excel(FILE, sheet_name='Tablib Dataset')
df_perf = pd.read_excel(FILE, sheet_name='performance')
df_area = pd.read_excel(FILE, sheet_name='área')

print(f'Tablib Dataset : {df_main.shape[0]:,} linhas x {df_main.shape[1]} colunas')
print(f'performance    : {df_perf.shape[0]:,} linhas x {df_perf.shape[1]} colunas')
print(f'área           : {df_area.shape[0]:,} linhas x {df_area.shape[1]} colunas')

Tablib Dataset : 4,281 linhas x 58 colunas
performance    : 2,382 linhas x 5 colunas
área           : 4,394 linhas x 2 colunas


---
## 1. Limpeza — Tablib Dataset

### 1.1 Remover coluna sem dados úteis

In [24]:
# 'Match' está 100% nula — sem valor analítico
print('Match — % nula:', df_main['Match'].isnull().mean() * 100, '%')
df_main = df_main.drop(columns=['Match'])

Match — % nula: 100.0 %


### 1.2 Remover colunas de URL (não analíticas)

In [25]:
url_cols = [c for c in df_main.columns if c.startswith('URL')]
print('Colunas URL removidas:', url_cols)
df_main = df_main.drop(columns=url_cols)

Colunas URL removidas: ['URL Raciocínio', 'URL Cultura', 'URL Social', 'URL Motivacional', 'URL Perfil']


### 1.3 Converter colunas de data/hora para datetime

In [26]:
date_cols = [c for c in df_main.columns if c.startswith('Início') or c.startswith('Fim')]

for col in date_cols:
    df_main[col] = pd.to_datetime(df_main[col], utc=True, errors='coerce')

print('Tipos das colunas de data:')
print(df_main[date_cols].dtypes)

Tipos das colunas de data:
Início - Raciocínio      datetime64[us, UTC]
Fim - Raciocínio         datetime64[us, UTC]
Início - Cultura         datetime64[us, UTC]
Fim - Cultura            datetime64[us, UTC]
Início - Social          datetime64[us, UTC]
Fim - Social             datetime64[us, UTC]
Início - Motivacional    datetime64[us, UTC]
Fim - Motivacional       datetime64[us, UTC]
Início - Perfil          datetime64[us, UTC]
Fim - Perfil             datetime64[us, UTC]
dtype: object


### 1.4 Tratar valores ausentes

As colunas de scores (`Potencial Bruto`, `Social`, `Motivacional`, `atributo-*`, `perfil-*`) possuem alta taxa de nulos pois representam testes que nem todos os candidatos realizaram. Esses nulos são **estruturais** — não há como imputá-los sem distorcer a análise. Registramos sua presença e os mantemos como `NaN`.

Para `Cultura pontuação` / `Cultura classificação` (~22,7% nulos) e `Raciocínio` (~24,5% nulos), aplica-se a mesma lógica.

In [27]:
# Criar flag binária indicando quais testes cada candidato realizou
testes = {
    'fez_raciocinio'   : 'Raciocínio',
    'fez_cultura'      : 'Cultura pontuação',
    'fez_social'       : 'Social',
    'fez_motivacional' : 'Motivacional',
    'fez_perfil'       : 'perfil-Comunicação',
    'fez_atributos'    : 'atributo-Comunicação',
}

for flag, col in testes.items():
    df_main[flag] = df_main[col].notna().astype(int)

print('Candidatos que realizaram cada teste:')
df_main[[*testes.keys()]].sum().to_frame('total').T

Candidatos que realizaram cada teste:


,fez_raciocinio,fez_cultura,fez_social,fez_motivacional,fez_perfil,fez_atributos
total,3234,3311,981,956,1019,921


### 1.5 Padronizar coluna de texto: `Cultura classificação`

In [28]:
import unicodedata

def remover_acentos(texto):
    if pd.isna(texto):
        return texto
    normalizado = unicodedata.normalize('NFKD', str(texto))
    return normalizado.encode('ascii', 'ignore').decode('ascii')

df_main['Cultura classificação'] = (
    df_main['Cultura classificação']
    .str.strip()
    .str.title()
    .map(remover_acentos)
    .str.replace(' ', '-', regex=False)
)

print(df_main['Cultura classificação'].value_counts(dropna=False))

Cultura classificação
NaN            970
Medio-Baixo    734
Baixo          671
Medio          636
Muito-Baixo    474
Medio-Alto     426
Alto           245
Muito-Alto     125
Name: count, dtype: int64


### 1.6 Padronizar nome: concatenar Nome + Sobrenome

In [29]:
df_main.insert(0, 'Nome Completo',
               (df_main['Nome'].str.strip() + ' ' + df_main['Sobrenome'].str.strip()).str.title())
df_main = df_main.drop(columns=['Nome', 'Sobrenome'])
df_main[['Nome Completo', 'E-mail', 'CPF']].head(3)

,Nome Completo,E-mail,CPF
0,Heloísa Duartche,heloisaduarte7623@example.com,582.975.481-98
1,Sophia Lima,sophialima1036@example.com,618.492.035-98
2,Cauã Alves,cauaalves8382@example.com,376.104.598-11


---
## 2. Limpeza — performance

### 2.1 Renomear colunas para padronizar

In [30]:
df_perf.columns = (
    df_perf.columns
        .str.replace('Performance ', 'perf_', regex=False)
        .str.replace('º/', '_', regex=False)
        .str.replace('º ', '_', regex=False)
)

df_perf.head(3)

,CPF,perf_1_2019,perf_2_2018,perf_1_2018,perf_2_2017
0,678.042.935-18,3.0,2.0,2.0,3.0
1,089.324.175-12,2.0,2.0,2.0,2.0
2,587.416.093-11,2.0,2.0,2.0,2.0


---
## 3. Limpeza — área

### 3.1 Padronizar categoria `Área`

In [31]:
df_area['Área'] = (
    df_area['Área']
    .str.strip()
    .str.title()
    .map(remover_acentos)
)

print(df_area['Área'].value_counts())

Área
Operacoes     1713
Comercial     1590
Logistica      919
Financeiro      92
Pessoas         80
Name: count, dtype: int64


### 3.2 Remover duplicatas de CPF em `área`

In [32]:
dup_area = df_area[df_area['CPF'].duplicated(keep=False)]
print(f'Linhas com CPF duplicado em área: {len(dup_area)}')

# Manter a última ocorrência (mais recente)
df_area = df_area.drop_duplicates(subset='CPF', keep='last').reset_index(drop=True)
print(f'Linhas após remoção de duplicatas: {len(df_area)}')

Linhas com CPF duplicado em área: 0
Linhas após remoção de duplicatas: 4394


---
## 4. Junção das Três Abas

Chave de junção: **CPF**

- `df_main` é o dataset principal (left)
- `df_perf` e `df_area` são enriquecimentos opcionais (left join)

In [35]:
df = (
    df_main
    .merge(df_area, on='CPF', how='left')
    .merge(df_perf,  on='CPF', how='left')
)

print(f'Dataset final: {df.shape[0]:,} linhas x {df.shape[1]} colunas\n')

cobertura_area = df['Área'].notna().sum()
print(f'Candidatos com área mapeada: {cobertura_area:,} ({cobertura_area/len(df)*100:.1f}%)\n')

print('Cobertura de performance por semestre:')
for col in [c for c in df.columns if c.startswith('perf_')]:
    n = df[col].notna().sum()
    print(f'  {col}: {n:,} ({n/len(df)*100:.1f}%)')

Dataset final: 4,281 linhas x 62 colunas

Candidatos com área mapeada: 4,281 (100.0%)

Cobertura de performance por semestre:
  perf_1_2019: 1,456 (34.0%)
  perf_2_2018: 1,486 (34.7%)
  perf_1_2018: 988 (23.1%)
  perf_2_2017: 353 (8.2%)


---
## 5. Resumo Final da Qualidade

In [36]:
print('Tipos de dados do dataset limpo:')
df.dtypes

Tipos de dados do dataset limpo:


Nome Completo          str
E-mail                 str
CPF                    str
Potencial Bruto    float64
Raciocínio         float64
                    ...   
Área                   str
perf_1_2019        float64
perf_2_2018        float64
perf_1_2018        float64
perf_2_2017        float64
Length: 62, dtype: object

---
## 6. Exportação

In [37]:
OUTPUT = 'dados_limpos.xlsx'

# Excel não suporta datetime com fuso horário — converter para UTC sem timezone
date_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
for col in date_cols:
    df[col] = df[col].dt.tz_convert('UTC').dt.tz_localize(None)

df.to_excel(OUTPUT, index=False)
print(f'Dataset exportado para "{OUTPUT}" — {df.shape[0]:,} linhas x {df.shape[1]} colunas')

Dataset exportado para "dados_limpos.xlsx" — 4,281 linhas x 62 colunas
